
<a id='rag'></a>

# Retrieval-Augmented Generation (RAG)

## RAG på norsk: Gjenfinningsforsterket tekstgenerering


## Språkmodellen

Vi kommer til å bruke modeller fra [Ollama](https://ollama.com/), en kjent plattform for modeller som kan brukes både på lokal maskin og i skyløsninger. I denne oppgaven vil vi bruke LLM [gemma3:1b](https://ollama.com/library/gemma3), som er en familie av modeller fra Google DeepMind. Dette er en liten modell med bare 1 milliard parametere. Det bør være mulig å bruke den på de fleste bærbare maskiner.

In [ ]:
pwd

In [ ]:
import os
# os.environ['HF_HOME'] = '/Users/ragnhildsundsbak/from-gutenberg-to-rstudio'
# vi må ha os på grunn av tokenet fra HF

import torch
device = 0 if torch.cuda.is_available() else -1

from langchain_community.llms import Ollama

llm = Ollama(
    model="mistral:latest"
)

query = 'What are the major contributions of the Trivandrum Observatory?'
output = llm.invoke(query)
print(output)

from langchain_ollama import OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(
    model="granite-embedding:latest",
)

document_folder = '/Users/ragnhildsundsbak/rtd-litteratur'

import os
from langchain_community.document_loaders import PyPDFLoader

document_folder = "/Users/ragnhildsundsbak/rtd-litteratur"

documents = []
for filename in os.listdir(document_folder):
    if filename.endswith(".pdf"):
        path = os.path.join(document_folder, filename)
        loader = PyPDFLoader(path)
        documents.extend(loader.load())

print(len(documents))
print(documents[0].metadata)
print(documents[0].page_content[:500])

print(f'Number of documents:', len(documents))
print('Maximum document length: ', max([len(doc.page_content) for doc in documents]))

print(documents[0])

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700, #  Could be more, for larger models like mistralai/Ministral-8B-Instruct-2410
    chunk_overlap  = 200,
)
documents = text_splitter.split_documents(documents)

print(f'Number of documents:', len(documents))
print('Maximum document length: ', max([len(doc.page_content) for doc in documents]))

from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents, ollama_embeddings)

relevant_documents = vectorstore.similarity_search(query)
print(f'Number of documents found: {len(relevant_documents)}')

print(relevant_documents[0].page_content)

retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

In [ ]:
from langchain_classic.prompts import PromptTemplate

prompt_template = '''You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
Context: {context}

Question: {input}

Answer:
'''

prompt = PromptTemplate(template=prompt_template,
                        input_variables=['context', 'input'])

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

combine_documents_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_documents_chain)

result = rag_chain.invoke({'input': query})

print(result['answer'])

Får du feilmeldinger? Lant ned [Sublime text](https://www.sublimetext.com/download) slik at du lettere får oversikt over koden din. Output filen gir et linjenummer for der feilen ligger. Da kan du lese av rette linjenummeret i Sublime editoren.